In [2]:
import pandas as pd
import numpy as np
from tqdm import tqdm

# ================= Configuration =================
MOVIES_FILE = 'movies.dat'
RATINGS_FILE = 'ratings.dat'
OUTPUT_FILE = 'movielens_sequential_dataset.csv'

# Filtering thresholds
MIN_ITEM_COUNT = 5       # Filter items appearing less than 5 times
MIN_SEQ_LEN = 20         # Filter users with sequence length less than 20

# Random seed for reproducibility
RANDOM_SEED = 42

In [3]:
def map_movielens_category(genre_str):
    """
    Maps MovieLens combined genres to a single Main Category based on specificity priority.
    Priority: Niche/Specific > Broad/Generic
    """
    if pd.isna(genre_str) or genre_str == '(no genres listed)':
        return 'Unknown'
    
    # Split the string into a list of genres
    genres = genre_str.split('|')
    
    # Priority list: from most specific/rare to most generic
    # You can adjust this order based on your research needs
    priority_map = [
        'Film-Noir', 'Documentary', 'War', 'Western', 'Musical', 
        'Animation', 'Children\'s', 'Fantasy', 'Sci-Fi', 'Horror', 
        'Crime', 'Thriller', 'Mystery', 'Action', 'Adventure', 
        'Romance', 'Comedy', 'Drama'
    ]
    
    # Return the first match in the priority list
    for p in priority_map:
        if p in genres:
            return p
            
    # Fallback if no specific genre matches (or for new genres)
    return genres[0]

print("Category mapping function defined.")

Category mapping function defined.


In [4]:
# --- Step 1: Load Movies Data ---
print("Loading movies data...")
# MovieLens uses '::' separator. engine='python' is required for multi-char separators.
movies_df = pd.read_csv(
    MOVIES_FILE, 
    sep='::', 
    header=None, 
    names=['MovieID', 'Title', 'Genres'], 
    engine='python',
    encoding='latin-1' # Common encoding for this dataset
)

# Apply category mapping
movies_df['category'] = movies_df['Genres'].apply(map_movielens_category)

# Create a lookup dictionary: MovieID (int) -> Category (str)
item_category_map = dict(zip(movies_df['MovieID'], movies_df['category']))

print(f"Loaded {len(movies_df)} movies.")
print("Sample Category Mapping:")
print(movies_df[['Title', 'Genres', 'category']].head())

Loading movies data...
Loaded 3883 movies.
Sample Category Mapping:
                                Title                        Genres  \
0                    Toy Story (1995)   Animation|Children's|Comedy   
1                      Jumanji (1995)  Adventure|Children's|Fantasy   
2             Grumpier Old Men (1995)                Comedy|Romance   
3            Waiting to Exhale (1995)                  Comedy|Drama   
4  Father of the Bride Part II (1995)                        Comedy   

     category  
0   Animation  
1  Children's  
2     Romance  
3      Comedy  
4      Comedy  


In [5]:
movies_df

,MovieID,Title,Genres,category
0,1,Toy Story (1995),Animation|Children's|Comedy,Animation
1,2,Jumanji (1995),Adventure|Children's|Fantasy,Children's
2,3,Grumpier Old Men (1995),Comedy|Romance,Romance
3,4,Waiting to Exhale (1995),Comedy|Drama,Comedy
4,5,Father of the Bride Part II (1995),Comedy,Comedy
...,...,...,...,...
3878,3948,Meet the Parents (2000),Comedy,Comedy
3879,3949,Requiem for a Dream (2000),Drama,Drama
3880,3950,Tigerland (2000),Drama,Drama
3881,3951,Two Family House (2000),Drama,Drama


In [6]:
import pandas as pd

# --- Step 2: Load Ratings Data ---
print("\nLoading ratings data...")
ratings_df = pd.read_csv(
    RATINGS_FILE, 
    sep='::', 
    header=None, 
    names=['UserID', 'MovieID', 'Rating', 'Timestamp'], 
    engine='python'
)

# We now keep UserID, MovieID, Rating, and Timestamp.
ratings_df = ratings_df[['UserID', 'MovieID', 'Rating', 'Timestamp']]

# --- Step 3: Filter Items (Count < 5) ---
print("Filtering unpopular items...")
item_counts = ratings_df['MovieID'].value_counts()
valid_items = item_counts[item_counts >= MIN_ITEM_COUNT].index

# Filter the dataframe
original_len = len(ratings_df)
ratings_df = ratings_df[ratings_df['MovieID'].isin(valid_items)].copy()

print(f"Original interactions: {original_len}")
print(f"After filtering items (<{MIN_ITEM_COUNT}): {len(ratings_df)}")
# You can now access ratings_df['Rating']
print(f"Columns present: {ratings_df.columns.tolist()}")


Loading ratings data...
Filtering unpopular items...
Original interactions: 1000209
After filtering items (<5): 999611
Columns present: ['UserID', 'MovieID', 'Rating', 'Timestamp']


In [7]:
# --- Step 4: Filter Sequences (Length < 20) ---
print("\nFiltering short sequences...")
user_counts = ratings_df['UserID'].value_counts()
valid_users = user_counts[user_counts >= MIN_SEQ_LEN].index

ratings_df = ratings_df[ratings_df['UserID'].isin(valid_users)].copy()
print(f"After filtering users (len < {MIN_SEQ_LEN}): {len(ratings_df)}")

# --- Step 5: ID Remapping (1...N) ---
print("Remapping User and Item IDs...")

# Sort by UserID and Timestamp to ensure order
ratings_df.sort_values(by=['UserID', 'Timestamp'], inplace=True)

# Create mappings
unique_users = ratings_df['UserID'].unique()
unique_items = ratings_df['MovieID'].unique()

user_map = {uid: i+1 for i, uid in enumerate(unique_users)}
item_map = {iid: i+1 for i, iid in enumerate(unique_items)}

# Apply mappings
ratings_df['user_id'] = ratings_df['UserID'].map(user_map)
ratings_df['item_id'] = ratings_df['MovieID'].map(item_map)
# Also rename Rating to rating for consistency
ratings_df['rating'] = ratings_df['Rating']

# Convert timestamp to datetime for consistency with Steam dataset
ratings_df['timestamp'] = pd.to_datetime(ratings_df['Timestamp'], unit='s')

# Map categories using original MovieID
ratings_df['category'] = ratings_df['MovieID'].map(item_category_map)

# Select final columns (Added 'rating' here)
final_df = ratings_df[['user_id', 'item_id', 'rating', 'timestamp', 'category']]

print(f"Final User count: {len(user_map)}")
print(f"Final Item count: {len(item_map)}")
print(final_df.head())


Filtering short sequences...
After filtering users (len < 20): 999517
Remapping User and Item IDs...
Final User count: 6035
Final Item count: 3416
    user_id  item_id  rating           timestamp category
31        1        1       4 2000-12-31 22:00:19    Drama
22        1        2       5 2000-12-31 22:00:55   Sci-Fi
27        1        3       4 2000-12-31 22:00:55  Romance
37        1        4       5 2000-12-31 22:00:55  Musical
24        1        5       3 2000-12-31 22:01:43  Romance


In [8]:
# --- Step 6: Statistics & Save ---
print("\n" + "="*40)
print("Dataset Statistics")
print("="*40)

# 1. Basic Counts
n_users = final_df['user_id'].nunique()
n_items = final_df['item_id'].nunique()
n_interactions = len(final_df)

# 2. Sequence Length Stats
seq_lens = final_df.groupby('user_id').size()
avg_seq_len = seq_lens.mean()

# 3. Category Diversity (Average unique categories per sequence)
# Group by user, get unique categories count
avg_unique_cats = final_df.groupby('user_id')['category'].nunique().mean()

print(f"Total Users: {n_users}")
print(f"Total Items: {n_items}")
print(f"Total Interactions: {n_interactions}")
print(f"Sparsity: {1 - (n_interactions / (n_users * n_items)):.5f}")
print("-" * 30)
print(f"Min Sequence Length: {seq_lens.min()}")
print(f"Max Sequence Length: {seq_lens.max()}")
print(f"Average Sequence Length: {avg_seq_len:.2f}")
print(f"Avg Unique Categories per User: {avg_unique_cats:.2f}")
print("="*40)

# Save to CSV
print(f"Saving to {OUTPUT_FILE}...")
final_df.to_csv(OUTPUT_FILE, index=False)
print("Done.")


Dataset Statistics
Total Users: 6035
Total Items: 3416
Total Interactions: 999517
Sparsity: 0.95152
------------------------------
Min Sequence Length: 20
Max Sequence Length: 2277
Average Sequence Length: 165.62
Avg Unique Categories per User: 13.97
Saving to movielens_sequential_dataset.csv...
Done.


In [9]:
# --- Step 5.5: Add Movie Title & Save to New File ---

NEW_OUTPUT_FILE = 'movielens_sequential_dataset_with_title.csv'

# 1. Create a lookup dictionary: MovieID (int) -> Title (str)
# utilizing the movies_df loaded in Step 1
item_title_map = dict(zip(movies_df['MovieID'], movies_df['Title']))

# 2. Map titles to the ratings dataframe using the original MovieID
ratings_df['title'] = ratings_df['MovieID'].map(item_title_map)

# 3. Select final columns including 'title'
final_df_with_title = ratings_df[['user_id', 'item_id', 'rating', 'timestamp', 'category', 'title']]

# 4. Preview and Save
print("New DataFrame Preview (with Title):")
print(final_df_with_title.head())

print(f"\nSaving to {NEW_OUTPUT_FILE}...")
final_df_with_title.to_csv(NEW_OUTPUT_FILE, index=False)
print("Done.")

New DataFrame Preview (with Title):
    user_id  item_id  rating           timestamp category  \
31        1        1       4 2000-12-31 22:00:19    Drama   
22        1        2       5 2000-12-31 22:00:55   Sci-Fi   
27        1        3       4 2000-12-31 22:00:55  Romance   
37        1        4       5 2000-12-31 22:00:55  Musical   
24        1        5       3 2000-12-31 22:01:43  Romance   

                        title  
31   Girl, Interrupted (1999)  
22  Back to the Future (1985)  
27             Titanic (1997)  
37          Cinderella (1950)  
24      Meet Joe Black (1998)  

Saving to movielens_sequential_dataset_with_title.csv...
Done.
